# Notebook 02: Data Quality and Profiling

## Objective

This notebook evaluates the quality of the PaySim financial transaction dataset using structural, statistical, and business-rule validation.

The notebook will:

- validate the source schema;
- check missing and duplicate records;
- validate categorical values;
- test numeric ranges;
- examine account identifier formats;
- evaluate transaction balance consistency;
- analyze fraud-related business rules;
- identify invalid or suspicious records;
- generate a reusable data-quality report.

No source records are modified in this notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "PS_20174392719_1491204439457_log.csv"
)

REPORT_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "gold"
    / "data_quality"
)

REPORT_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(RAW_DATA_PATH)
print(REPORT_OUTPUT_PATH)

c:\Projects\paysim-financial-data-pipeline\data\raw\PS_20174392719_1491204439457_log.csv
c:\Projects\paysim-financial-data-pipeline\data\gold\data_quality


In [4]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "PS_20174392719_1491204439457_log.csv"
)

REPORT_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "gold"
    / "data_quality"
)

REPORT_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(RAW_DATA_PATH)
print(REPORT_OUTPUT_PATH)

c:\Projects\paysim-financial-data-pipeline\data\raw\PS_20174392719_1491204439457_log.csv
c:\Projects\paysim-financial-data-pipeline\data\gold\data_quality


In [5]:
EXPECTED_DTYPES = {
    "step": "int32",
    "type": "string",
    "amount": "float64",
    "nameOrig": "string",
    "oldbalanceOrg": "float64",
    "newbalanceOrig": "float64",
    "nameDest": "string",
    "oldbalanceDest": "float64",
    "newbalanceDest": "float64",
    "isFraud": "int8",
    "isFlaggedFraud": "int8",
}

df = pd.read_csv(
    RAW_DATA_PATH,
    dtype=EXPECTED_DTYPES,
)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Rows: 6,362,620
Columns: 11


In [6]:
memory_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"Memory usage: {memory_mb:,.2f} MB")

Memory usage: 597.00 MB


In [7]:
EXPECTED_COLUMNS = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud",
    "isFlaggedFraud",
]

EXPECTED_COLUMNS

['step',
 'type',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud']

In [8]:
actual_columns = df.columns.tolist()

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(actual_columns))
unexpected_columns = sorted(set(actual_columns) - set(EXPECTED_COLUMNS))
column_order_matches = actual_columns == EXPECTED_COLUMNS

print("Missing columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)
print("Column order matches:", column_order_matches)

Missing columns: []
Unexpected columns: []
Column order matches: True


In [9]:
quality_results = []


def add_quality_result(
    check_name: str,
    check_category: str,
    failed_count: int,
    total_count: int,
    severity: str = "ERROR",
    description: str = "",
) -> None:
    """
    Add one validation result to the data-quality report.
    """
    failure_rate = (
        failed_count / total_count * 100
        if total_count > 0
        else 0.0
    )

    quality_results.append(
        {
            "check_name": check_name,
            "check_category": check_category,
            "description": description,
            "severity": severity,
            "records_tested": total_count,
            "failed_records": failed_count,
            "failure_rate_pct": failure_rate,
            "status": "PASS" if failed_count == 0 else "FAIL",
        }
    )

In [11]:
add_quality_result(
    check_name="expected_columns_present",
    check_category="schema",
    failed_count=len(missing_columns),
    total_count=len(EXPECTED_COLUMNS),
    description="All required source columns must be present.",
)

In [12]:
add_quality_result(
    check_name="no_unexpected_columns",
    check_category="schema",
    failed_count=len(unexpected_columns),
    total_count=len(actual_columns),
    severity="WARNING",
    description="The source should not contain undocumented columns.",
)

In [13]:
add_quality_result(
    check_name="expected_column_order",
    check_category="schema",
    failed_count=0 if column_order_matches else 1,
    total_count=1,
    severity="WARNING",
    description="Source column order should match the documented schema.",
)

In [14]:
missing_summary = (
    df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"] / len(df) * 100
)

missing_summary

,missing_count,missing_pct
step,0,0.0
type,0,0.0
amount,0,0.0
nameOrig,0,0.0
oldbalanceOrg,0,0.0
newbalanceOrig,0,0.0
nameDest,0,0.0
oldbalanceDest,0,0.0
newbalanceDest,0,0.0
isFraud,0,0.0


In [15]:
for column in df.columns:
    failed_count = int(df[column].isna().sum())

    add_quality_result(
        check_name=f"{column}_not_null",
        check_category="completeness",
        failed_count=failed_count,
        total_count=len(df),
        description=f"{column} should not contain null values.",
    )

In [16]:
duplicate_count = int(df.duplicated().sum())

print(f"Exact duplicate rows: {duplicate_count:,}")

Exact duplicate rows: 0


In [17]:
add_quality_result(
    check_name="no_exact_duplicate_rows",
    check_category="uniqueness",
    failed_count=duplicate_count,
    total_count=len(df),
    description="The source should not contain exact duplicate transactions.",
)

In [18]:
VALID_TRANSACTION_TYPES = {
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER",
}

observed_transaction_types = set(df["type"].dropna().unique())

invalid_transaction_types = (
    observed_transaction_types - VALID_TRANSACTION_TYPES
)

print("Observed types:", observed_transaction_types)
print("Invalid types:", invalid_transaction_types)

Observed types: {'DEBIT', 'PAYMENT', 'TRANSFER', 'CASH_IN', 'CASH_OUT'}
Invalid types: set()


In [19]:
invalid_type_mask = ~df["type"].isin(VALID_TRANSACTION_TYPES)

invalid_type_count = int(invalid_type_mask.sum())

add_quality_result(
    check_name="valid_transaction_type",
    check_category="validity",
    failed_count=invalid_type_count,
    total_count=len(df),
    description="Transaction type must belong to the documented domain.",
)

In [20]:
df.loc[invalid_type_mask].head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud


In [21]:
step_summary = {
    "minimum": int(df["step"].min()),
    "maximum": int(df["step"].max()),
    "unique_steps": int(df["step"].nunique()),
}

step_summary

{'minimum': 1, 'maximum': 743, 'unique_steps': 743}

In [22]:
MIN_STEP = 1
MAX_STEP = 744

invalid_step_mask = ~df["step"].between(MIN_STEP, MAX_STEP)

invalid_step_count = int(invalid_step_mask.sum())

add_quality_result(
    check_name="valid_step_range",
    check_category="validity",
    failed_count=invalid_step_count,
    total_count=len(df),
    description="Simulation step must be between 1 and 744.",
)

In [23]:
expected_steps = set(range(MIN_STEP, MAX_STEP + 1))
observed_steps = set(df["step"].unique())

missing_steps = sorted(expected_steps - observed_steps)

missing_steps

[744]

In [24]:
add_quality_result(
    check_name="all_expected_steps_present",
    check_category="completeness",
    failed_count=len(missing_steps),
    total_count=MAX_STEP,
    severity="WARNING",
    description="All documented hourly simulation steps should be represented.",
)

In [25]:
df["amount"].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    6.362620e+06
mean     1.798619e+05
std      6.038582e+05
min      0.000000e+00
1%       4.494676e+02
25%      1.338957e+04
50%      7.487194e+04
75%      2.087215e+05
95%      5.186342e+05
99%      1.615979e+06
max      9.244552e+07
Name: amount, dtype: float64

In [26]:
negative_amount_mask = df["amount"] < 0
negative_amount_count = int(negative_amount_mask.sum())

add_quality_result(
    check_name="non_negative_amount",
    check_category="validity",
    failed_count=negative_amount_count,
    total_count=len(df),
    description="Transaction amount must not be negative.",
)

In [27]:
zero_amount_mask = df["amount"] == 0
zero_amount_count = int(zero_amount_mask.sum())

print(f"Zero-value transactions: {zero_amount_count:,}")

Zero-value transactions: 16


In [28]:
add_quality_result(
    check_name="non_zero_amount",
    check_category="business_rule",
    failed_count=zero_amount_count,
    total_count=len(df),
    severity="WARNING",
    description="Zero-value transactions require review but are not automatically rejected.",
)

In [29]:
numeric_columns = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

for column in numeric_columns:
    infinite_count = int(np.isinf(df[column]).sum())

    add_quality_result(
        check_name=f"{column}_finite",
        check_category="validity",
        failed_count=infinite_count,
        total_count=len(df),
        description=f"{column} must contain finite numeric values.",
    )

In [30]:
BALANCE_COLUMNS = [
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

for column in BALANCE_COLUMNS:
    negative_count = int((df[column] < 0).sum())

    add_quality_result(
        check_name=f"{column}_non_negative",
        check_category="validity",
        failed_count=negative_count,
        total_count=len(df),
        description=f"{column} should not contain negative balances.",
    )

In [31]:
VALID_BINARY_VALUES = {0, 1}

for column in ["isFraud", "isFlaggedFraud"]:
    invalid_flag_mask = ~df[column].isin(VALID_BINARY_VALUES)
    invalid_flag_count = int(invalid_flag_mask.sum())

    add_quality_result(
        check_name=f"{column}_binary",
        check_category="validity",
        failed_count=invalid_flag_count,
        total_count=len(df),
        description=f"{column} must contain only 0 or 1.",
    )

In [32]:
invalid_origin_mask = ~df["nameOrig"].str.match(r"^C\d+$", na=False)
invalid_origin_count = int(invalid_origin_mask.sum())

add_quality_result(
    check_name="valid_origin_account_format",
    check_category="validity",
    failed_count=invalid_origin_count,
    total_count=len(df),
    description="Origin account identifiers should follow the C<number> pattern.",
)

In [33]:
df.loc[
    invalid_origin_mask,
    ["nameOrig", "type", "amount"]
].head()

,nameOrig,type,amount


In [34]:
valid_destination_mask = df["nameDest"].str.match(
    r"^[CM]\d+$",
    na=False,
)

invalid_destination_count = int((~valid_destination_mask).sum())

add_quality_result(
    check_name="valid_destination_account_format",
    check_category="validity",
    failed_count=invalid_destination_count,
    total_count=len(df),
    description="Destination identifiers should follow C<number> or M<number>.",
)

In [35]:
self_transfer_mask = df["nameOrig"] == df["nameDest"]
self_transfer_count = int(self_transfer_mask.sum())

print(f"Self-transfers: {self_transfer_count:,}")

Self-transfers: 0


In [36]:
add_quality_result(
    check_name="no_self_transfer",
    check_category="business_rule",
    failed_count=self_transfer_count,
    total_count=len(df),
    severity="WARNING",
    description="Origin and destination should normally be different accounts.",
)

In [37]:
merchant_mask = df["nameDest"].str.startswith("M", na=False)

merchant_balance_summary = df.loc[
    merchant_mask,
    ["oldbalanceDest", "newbalanceDest"]
].describe()

merchant_balance_summary

,oldbalanceDest,newbalanceDest
count,2151495.0,2151495.0
mean,0.0,0.0
std,0.0,0.0
min,0.0,0.0
25%,0.0,0.0
50%,0.0,0.0
75%,0.0,0.0
max,0.0,0.0


In [38]:
merchant_nonzero_balance_mask = (
    merchant_mask
    & (
        (df["oldbalanceDest"] != 0)
        | (df["newbalanceDest"] != 0)
    )
)

merchant_nonzero_balance_count = int(
    merchant_nonzero_balance_mask.sum()
)

add_quality_result(
    check_name="merchant_destination_balance_unavailable",
    check_category="business_rule",
    failed_count=merchant_nonzero_balance_count,
    total_count=int(merchant_mask.sum()),
    severity="WARNING",
    description=(
        "Merchant destinations are expected to have unavailable balances, "
        "typically represented by zeros."
    ),
)

In [39]:
outgoing_types = [
    "PAYMENT",
    "TRANSFER",
    "CASH_OUT",
    "DEBIT",
]

origin_outgoing_mask = df["type"].isin(outgoing_types)

df["origin_balance_error"] = (
    df["oldbalanceOrg"]
    - df["amount"]
    - df["newbalanceOrig"]
).abs()

In [40]:
BALANCE_TOLERANCE = 0.01

origin_balance_mismatch_mask = (
    origin_outgoing_mask
    & (df["origin_balance_error"] > BALANCE_TOLERANCE)
)

origin_balance_mismatch_count = int(
    origin_balance_mismatch_mask.sum()
)

print(
    "Outgoing origin balance mismatches:",
    f"{origin_balance_mismatch_count:,}",
)

Outgoing origin balance mismatches: 3,678,407


In [41]:
add_quality_result(
    check_name="outgoing_origin_balance_reconciliation",
    check_category="consistency",
    failed_count=origin_balance_mismatch_count,
    total_count=int(origin_outgoing_mask.sum()),
    severity="WARNING",
    description=(
        "Outgoing transaction balances should approximately reconcile, "
        "but PaySim may contain documented inconsistencies."
    ),
)

In [42]:
cash_in_mask = df["type"] == "CASH_IN"

df["cash_in_origin_balance_error"] = (
    df["oldbalanceOrg"]
    + df["amount"]
    - df["newbalanceOrig"]
).abs()

cash_in_mismatch_mask = (
    cash_in_mask
    & (
        df["cash_in_origin_balance_error"]
        > BALANCE_TOLERANCE
    )
)

cash_in_mismatch_count = int(cash_in_mismatch_mask.sum())

add_quality_result(
    check_name="cash_in_origin_balance_reconciliation",
    check_category="consistency",
    failed_count=cash_in_mismatch_count,
    total_count=int(cash_in_mask.sum()),
    severity="WARNING",
    description="Cash-in origin balances should approximately reconcile.",
)

In [43]:
fraud_by_type = (
    df.groupby("type", observed=True)
    .agg(
        transaction_count=("type", "size"),
        fraud_count=("isFraud", "sum"),
        total_amount=("amount", "sum"),
    )
    .reset_index()
)

fraud_by_type["fraud_rate_pct"] = (
    fraud_by_type["fraud_count"]
    / fraud_by_type["transaction_count"]
    * 100
)

fraud_by_type.sort_values(
    "fraud_rate_pct",
    ascending=False,
)

,type,transaction_count,fraud_count,total_amount,fraud_rate_pct
4,TRANSFER,532909,4097,4.852920e+11,0.768799
1,CASH_OUT,2237500,4116,3.944130e+11,0.183955
0,CASH_IN,1399284,0,2.363674e+11,0.0
2,DEBIT,41432,0,2.271992e+08,0.0
3,PAYMENT,2151495,0,2.809337e+10,0.0


In [44]:
expected_fraud_types = {"TRANSFER", "CASH_OUT"}

unexpected_fraud_mask = (
    (df["isFraud"] == 1)
    & (~df["type"].isin(expected_fraud_types))
)

unexpected_fraud_count = int(unexpected_fraud_mask.sum())

add_quality_result(
    check_name="fraud_transaction_type_consistency",
    check_category="business_rule",
    failed_count=unexpected_fraud_count,
    total_count=int((df["isFraud"] == 1).sum()),
    severity="WARNING",
    description=(
        "Fraudulent PaySim transactions are expected primarily in "
        "TRANSFER and CASH_OUT records."
    ),
)

In [45]:
FLAG_THRESHOLD = 200_000

flag_summary = (
    df.groupby(
        ["type", "isFlaggedFraud"],
        observed=True,
    )
    .agg(
        transaction_count=("amount", "size"),
        minimum_amount=("amount", "min"),
        maximum_amount=("amount", "max"),
        average_amount=("amount", "mean"),
    )
    .reset_index()
)

flag_summary

,type,isFlaggedFraud,transaction_count,minimum_amount,maximum_amount,average_amount
0,CASH_IN,0,1399284,0.04,1915267.90,1.689202e+05
1,CASH_OUT,0,2237500,0.00,10000000.00,1.762740e+05
2,DEBIT,0,41432,0.55,569077.51,5.483665e+03
3,PAYMENT,0,2151495,0.02,238637.98,1.305760e+04
4,TRANSFER,0,532893,2.60,92445516.64,9.105284e+05
5,TRANSFER,1,16,353874.22,10000000.00,4.861598e+06


In [46]:
invalid_flag_rule_mask = (
    (df["isFlaggedFraud"] == 1)
    & (
        (df["type"] != "TRANSFER")
        | (df["amount"] <= FLAG_THRESHOLD)
    )
)

invalid_flag_rule_count = int(
    invalid_flag_rule_mask.sum()
)

add_quality_result(
    check_name="flagged_fraud_threshold_rule",
    check_category="business_rule",
    failed_count=invalid_flag_rule_count,
    total_count=int((df["isFlaggedFraud"] == 1).sum()),
    severity="WARNING",
    description=(
        "Flagged fraud should represent transfers above the documented "
        "200,000 threshold."
    ),
)

In [47]:
unflagged_high_value_transfer_mask = (
    (df["type"] == "TRANSFER")
    & (df["amount"] > FLAG_THRESHOLD)
    & (df["isFlaggedFraud"] == 0)
)

unflagged_high_value_transfer_count = int(
    unflagged_high_value_transfer_mask.sum()
)

print(
    "High-value transfers not flagged:",
    f"{unflagged_high_value_transfer_count:,}",
)

High-value transfers not flagged: 409,094


In [48]:
add_quality_result(
    check_name="high_value_transfer_flag_coverage",
    check_category="business_rule",
    failed_count=unflagged_high_value_transfer_count,
    total_count=int(
        (
            (df["type"] == "TRANSFER")
            & (df["amount"] > FLAG_THRESHOLD)
        ).sum()
    ),
    severity="WARNING",
    description=(
        "Measures how many transfers above 200,000 were not flagged."
    ),
)

In [49]:
fraud_flag_crosstab = pd.crosstab(
    df["isFraud"],
    df["isFlaggedFraud"],
    margins=True,
)

fraud_flag_crosstab

isFlaggedFraud,0,1,All
isFraud,,,
0,6354407,0,6354407
1,8197,16,8213
All,6362604,16,6362620


In [50]:
fraudulent_count = int((df["isFraud"] == 1).sum())

flagged_and_fraud_count = int(
    (
        (df["isFraud"] == 1)
        & (df["isFlaggedFraud"] == 1)
    ).sum()
)

flag_detection_rate = (
    flagged_and_fraud_count / fraudulent_count * 100
    if fraudulent_count > 0
    else 0
)

print(f"Fraud detection rate of business flag: {flag_detection_rate:.4f}%")

Fraud detection rate of business flag: 0.1948%


In [51]:
numerical_profile = df[
    [
        "step",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest",
    ]
].describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
    ]
).T

numerical_profile

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
step,6362620.0,2.433972e+02,1.423320e+02,1.0,9.0000,16.0000,156.00,239.000,3.350000e+02,4.900000e+02,6.810000e+02,7.430000e+02
amount,6362620.0,1.798619e+05,6.038582e+05,0.0,449.4676,2224.0995,13389.57,74871.940,2.087215e+05,5.186342e+05,1.615979e+06,9.244552e+07
oldbalanceOrg,6362620.0,8.338831e+05,2.888243e+06,0.0,0.0000,0.0000,0.00,14208.000,1.073152e+05,5.823702e+06,1.602726e+07,5.958504e+07
newbalanceOrig,6362620.0,8.551137e+05,2.924049e+06,0.0,0.0000,0.0000,0.00,0.000,1.442584e+05,5.980262e+06,1.617616e+07,4.958504e+07
oldbalanceDest,6362620.0,1.100702e+06,3.399180e+06,0.0,0.0000,0.0000,0.00,132705.665,9.430367e+05,5.147230e+06,1.237182e+07,3.560159e+08
newbalanceDest,6362620.0,1.224996e+06,3.674129e+06,0.0,0.0000,0.0000,0.00,214661.440,1.111909e+06,5.515716e+06,1.313787e+07,3.561793e+08


In [52]:
cardinality_profile = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "unique_count": [
            df[column].nunique(dropna=False)
            for column in df.columns
        ],
    }
)

cardinality_profile["cardinality_pct"] = (
    cardinality_profile["unique_count"]
    / len(df)
    * 100
)

cardinality_profile.sort_values(
    "cardinality_pct",
    ascending=False,
)

,column,dtype,unique_count,cardinality_pct
3,nameOrig,string,6353307,99.853629
2,amount,float64,5316900,83.564632
11,origin_balance_error,float64,4716888,74.134366
12,cash_in_origin_balance_error,float64,4565517,71.755299
7,oldbalanceDest,float64,3614697,56.811455
8,newbalanceDest,float64,3555499,55.881052
6,nameDest,string,2722362,42.786808
5,newbalanceOrig,float64,2682586,42.161657
4,oldbalanceOrg,float64,1845844,29.010753
0,step,int32,743,0.011678


In [53]:
transaction_type_profile = (
    df.groupby("type", observed=True)
    .agg(
        transaction_count=("type", "size"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean"),
        median_amount=("amount", "median"),
        maximum_amount=("amount", "max"),
        fraud_count=("isFraud", "sum"),
        flagged_fraud_count=("isFlaggedFraud", "sum"),
    )
    .reset_index()
)

transaction_type_profile["transaction_pct"] = (
    transaction_type_profile["transaction_count"]
    / len(df)
    * 100
)

transaction_type_profile["fraud_rate_pct"] = (
    transaction_type_profile["fraud_count"]
    / transaction_type_profile["transaction_count"]
    * 100
)

transaction_type_profile

,type,transaction_count,total_amount,average_amount,median_amount,maximum_amount,fraud_count,flagged_fraud_count,transaction_pct,fraud_rate_pct
0,CASH_IN,1399284,2.363674e+11,168920.242004,143427.710,1915267.90,0,0,21.992261,0.0
1,CASH_OUT,2237500,3.944130e+11,176273.964346,147072.185,10000000.00,4116,0,35.166331,0.183955
2,DEBIT,41432,2.271992e+08,5483.665314,3048.990,569077.51,0,0,0.651178,0.0
3,PAYMENT,2151495,2.809337e+10,13057.604660,9482.190,238637.98,0,0,33.814608,0.0
4,TRANSFER,532909,4.852920e+11,910647.009645,486308.390,92445516.64,4097,16,8.375622,0.768799


In [54]:
transaction_type_profile = (
    df.groupby("type", observed=True)
    .agg(
        transaction_count=("type", "size"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean"),
        median_amount=("amount", "median"),
        maximum_amount=("amount", "max"),
        fraud_count=("isFraud", "sum"),
        flagged_fraud_count=("isFlaggedFraud", "sum"),
    )
    .reset_index()
)

transaction_type_profile["transaction_pct"] = (
    transaction_type_profile["transaction_count"]
    / len(df)
    * 100
)

transaction_type_profile["fraud_rate_pct"] = (
    transaction_type_profile["fraud_count"]
    / transaction_type_profile["transaction_count"]
    * 100
)

transaction_type_profile

,type,transaction_count,total_amount,average_amount,median_amount,maximum_amount,fraud_count,flagged_fraud_count,transaction_pct,fraud_rate_pct
0,CASH_IN,1399284,2.363674e+11,168920.242004,143427.710,1915267.90,0,0,21.992261,0.0
1,CASH_OUT,2237500,3.944130e+11,176273.964346,147072.185,10000000.00,4116,0,35.166331,0.183955
2,DEBIT,41432,2.271992e+08,5483.665314,3048.990,569077.51,0,0,0.651178,0.0
3,PAYMENT,2151495,2.809337e+10,13057.604660,9482.190,238637.98,0,0,33.814608,0.0
4,TRANSFER,532909,4.852920e+11,910647.009645,486308.390,92445516.64,4097,16,8.375622,0.768799


In [56]:
hourly_profile = (
    df.groupby("step")
    .agg(
        transaction_count=("step", "size"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean"),
        fraud_count=("isFraud", "sum"),
        flagged_fraud_count=("isFlaggedFraud", "sum"),
    )
    .reset_index()
)

hourly_profile["fraud_rate_pct"] = (
    hourly_profile["fraud_count"]
    / hourly_profile["transaction_count"]
    * 100
)

hourly_profile.head()

,step,transaction_count,total_amount,average_amount,fraud_count,flagged_fraud_count,fraud_rate_pct
0,1,2708,2.854292e+08,105402.208696,16,0,0.590842
1,2,1014,8.592160e+07,84735.309684,8,0,0.788955
2,3,552,4.329388e+07,78430.950036,4,0,0.724638
3,4,565,7.291003e+07,129044.298354,10,0,1.769912
4,5,665,4.554809e+07,68493.368045,6,0,0.902256


In [57]:
hourly_profile[
    [
        "transaction_count",
        "total_amount",
        "fraud_count",
        "fraud_rate_pct",
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
transaction_count,743.0,8.563419e+03,1.338869e+04,2.0,1.200000e+01,5.290000e+02,1.231700e+04,5.135200e+04
total_amount,743.0,1.540233e+09,2.665957e+09,84837.7,1.372773e+07,7.468351e+07,1.855992e+09,2.496355e+10
fraud_count,743.0,1.105384e+01,4.998631e+00,0.0,8.000000e+00,1.000000e+01,1.400000e+01,4.000000e+01
fraud_rate_pct,743.0,4.392526e+01,4.895619e+01,0.0,7.489479e-02,1.915709e+00,1.000000e+02,1.000000e+02


In [58]:
quality_report = pd.DataFrame(quality_results)

quality_report = quality_report[
    [
        "check_name",
        "check_category",
        "description",
        "severity",
        "records_tested",
        "failed_records",
        "failure_rate_pct",
        "status",
    ]
]

quality_report.sort_values(
    by=["status", "severity", "check_category", "check_name"],
    ascending=[True, True, True, True],
)

,check_name,check_category,description,severity,records_tested,failed_records,failure_rate_pct,status
39,high_value_transfer_flag_coverage,business_rule,"Measures how many transfers above 200,000 were...",WARNING,409110,409094,99.996089,FAIL
19,non_zero_amount,business_rule,Zero-value transactions require review but are...,WARNING,6362620,16,0.000251,FAIL
17,all_expected_steps_present,completeness,All documented hourly simulation steps should ...,WARNING,744,1,0.134409,FAIL
36,cash_in_origin_balance_reconciliation,consistency,Cash-in origin balances should approximately r...,WARNING,1399284,101096,7.224838,FAIL
35,outgoing_origin_balance_reconciliation,consistency,Outgoing transaction balances should approxima...,WARNING,4963336,3678407,74.111585,FAIL
5,amount_not_null,completeness,amount should not contain null values.,ERROR,6362620,0,0.000000,PASS
13,isFlaggedFraud_not_null,completeness,isFlaggedFraud should not contain null values.,ERROR,6362620,0,0.000000,PASS
12,isFraud_not_null,completeness,isFraud should not contain null values.,ERROR,6362620,0,0.000000,PASS
9,nameDest_not_null,completeness,nameDest should not contain null values.,ERROR,6362620,0,0.000000,PASS
6,nameOrig_not_null,completeness,nameOrig should not contain null values.,ERROR,6362620,0,0.000000,PASS


In [59]:
quality_report["status"].value_counts()

status
PASS    35
FAIL     5
Name: count, dtype: int64

In [60]:
quality_report.groupby(
    ["check_category", "status"]
).size().unstack(fill_value=0)

status,FAIL,PASS
check_category,,
business_rule,2,4
completeness,1,11
consistency,2,0
schema,0,3
uniqueness,0,1
validity,0,16


In [61]:
failed_checks = quality_report.loc[
    quality_report["status"] == "FAIL"
].sort_values(
    "failure_rate_pct",
    ascending=False,
)

failed_checks

,check_name,check_category,description,severity,records_tested,failed_records,failure_rate_pct,status
39,high_value_transfer_flag_coverage,business_rule,"Measures how many transfers above 200,000 were...",WARNING,409110,409094,99.996089,FAIL
35,outgoing_origin_balance_reconciliation,consistency,Outgoing transaction balances should approxima...,WARNING,4963336,3678407,74.111585,FAIL
36,cash_in_origin_balance_reconciliation,consistency,Cash-in origin balances should approximately r...,WARNING,1399284,101096,7.224838,FAIL
17,all_expected_steps_present,completeness,All documented hourly simulation steps should ...,WARNING,744,1,0.134409,FAIL
19,non_zero_amount,business_rule,Zero-value transactions require review but are...,WARNING,6362620,16,0.000251,FAIL


## Validation Strategy

Data-quality checks are divided into two categories.

### Hard validation failures

Records may be quarantined when they contain:

- missing critical fields;
- invalid transaction types;
- negative transaction amounts;
- invalid fraud indicators;
- invalid simulation steps;
- malformed account identifiers.

### Soft validation warnings

Records are retained but monitored when they contain:

- zero-value transactions;
- balance reconciliation mismatches;
- missing expected hourly steps;
- self-transfers;
- inconsistent fraud-flag behavior;
- unusual merchant balance values.

This approach avoids silently dropping valid but unusual financial records.

In [62]:
hard_failure_masks = {
    "missing_critical_value": df[EXPECTED_COLUMNS].isna().any(axis=1),
    "invalid_transaction_type": ~df["type"].isin(
        VALID_TRANSACTION_TYPES
    ),
    "invalid_step": ~df["step"].between(MIN_STEP, MAX_STEP),
    "negative_amount": df["amount"] < 0,
    "invalid_fraud_flag": ~df["isFraud"].isin({0, 1}),
    "invalid_flagged_fraud": ~df["isFlaggedFraud"].isin({0, 1}),
    "invalid_origin_account": ~df["nameOrig"].str.match(
        r"^C\d+$",
        na=False,
    ),
    "invalid_destination_account": ~df["nameDest"].str.match(
        r"^[CM]\d+$",
        na=False,
    ),
}

In [63]:
hard_failure_frame = pd.DataFrame(hard_failure_masks)

df["has_hard_failure"] = hard_failure_frame.any(axis=1)

print(
    "Valid records:",
    f"{(~df['has_hard_failure']).sum():,}",
)

print(
    "Quarantine candidates:",
    f"{df['has_hard_failure'].sum():,}",
)

Valid records: 6,362,620
Quarantine candidates: 0


In [64]:
def build_failure_reason(row: pd.Series) -> str:
    reasons = [
        reason
        for reason, failed in row.items()
        if failed
    ]

    return "|".join(reasons)


failure_reasons = hard_failure_frame.loc[
    hard_failure_frame.any(axis=1)
].apply(
    build_failure_reason,
    axis=1,
)

failure_reasons.head()

Series([], dtype: float64)

In [65]:
quality_report.to_csv(
    REPORT_OUTPUT_PATH / "quality_report.csv",
    index=False,
)

missing_summary.to_csv(
    REPORT_OUTPUT_PATH / "missing_value_summary.csv",
)

cardinality_profile.to_csv(
    REPORT_OUTPUT_PATH / "cardinality_profile.csv",
    index=False,
)

transaction_type_profile.to_csv(
    REPORT_OUTPUT_PATH / "transaction_type_profile.csv",
    index=False,
)

hourly_profile.to_csv(
    REPORT_OUTPUT_PATH / "hourly_profile.csv",
    index=False,
)

fraud_by_type.to_csv(
    REPORT_OUTPUT_PATH / "fraud_by_transaction_type.csv",
    index=False,
)

In [66]:
sorted(
    path.name
    for path in REPORT_OUTPUT_PATH.iterdir()
)

['cardinality_profile.csv',
 'fraud_by_transaction_type.csv',
 'hourly_profile.csv',
 'missing_value_summary.csv',
 'quality_report.csv',
 'transaction_type_profile.csv']

## Final Findings

### Structural quality

- The source contains the expected 11 columns.
- Critical columns have no missing values.
- Column types are compatible with the documented schema.
- Exact duplicate records were evaluated.

### Domain validity

- Transaction types were checked against the five documented values.
- Fraud indicators were checked for binary values.
- Account identifiers were checked against customer and merchant patterns.
- Transaction steps were checked against the documented simulation range.

### Business-rule findings

- Merchant destination balances were evaluated separately because merchant
  balance information is unavailable.
- Transaction balance arithmetic does not necessarily reconcile for all
  records and is treated as a warning rather than a hard rejection rule.
- The business fraud flag captures only a narrow subset of fraudulent
  activity and should not be interpreted as a complete fraud detector.

### Pipeline implications

- Hard validation failures will be routed to a quarantine layer.
- Soft consistency warnings will remain in the Silver layer with monitoring
  metrics.
- Raw fields will be preserved in Bronze.
- Balance columns will remain available for audit but will be excluded from
  the downstream fraud-model feature table.